# SolarLoss scientific audit — La Réunion rooftop PV

This notebook downloads pinned Zenodo v2 source files, verifies their published byte counts and MD5 digests, audits measurements and timestamps, evaluates chronological healthy-period baselines, and writes reproducible JSON outputs.

Resource note: the electrical CSVs total about 261 MB; the optional weather file is 1.54 GB. Raw CSV audits and weather aggregation are chunked. Compact electrical columns are retained for modeling and may need several GB of RAM. In Colab, use a high-RAM runtime and persistent Drive storage. Set INCLUDE_WEATHER=False to skip the large weather download and weather-assisted candidate.

No fault labels are predictors. Training and selection use healthy periods only. Results describe measured deviations from a healthy-period reference; they do not prove cause, commercial recovery, Indian-site performance, or general model accuracy.


In [ ]:
import os
import sys
import subprocess
from pathlib import Path

configured = os.environ.get("SOLARLOSS_ANALYSIS_DIR")
candidates = [Path(configured)] if configured else []
for base in [Path.cwd(), *Path.cwd().parents]:
    candidates.extend([base / "analysis", base])
ANALYSIS_DIR = next((p.resolve() for p in candidates if (p / "requirements.txt").is_file()), None)
if ANALYSIS_DIR is None:
    raise FileNotFoundError("Extract the SolarLoss analysis bundle or set SOLARLOSS_ANALYSIS_DIR.")
print("Analysis directory:", ANALYSIS_DIR)

# Pinned, reproducible dependencies for Colab or local Jupyter.
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", str(ANALYSIS_DIR / "requirements.txt")])


In [ ]:
sys.path.insert(0, str(ANALYSIS_DIR / "src"))
from solarloss_research.dataset import FILES, download_file
from solarloss_research.pipeline import audit_and_model, export_results
from solarloss_research.model import _predict, _model_frame
from datetime import datetime, timezone
from IPython.display import display
import pandas as pd

try:
    from google.colab import drive
    drive.mount("/content/drive")
    DEFAULT_ROOT = Path("/content/drive/MyDrive/solarloss_research")
except ImportError:
    DEFAULT_ROOT = ANALYSIS_DIR

DATA_DIR = Path(os.environ.get("SOLARLOSS_DATA_DIR", DEFAULT_ROOT / "data"))
RESULTS_DIR = Path(os.environ.get("SOLARLOSS_RESULTS_DIR", DEFAULT_ROOT / "results"))
INCLUDE_WEATHER = True  # Set False to omit the 1.54 GB weather file and candidate.
CHUNKSIZE = 250_000
DATA_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print("Data:", DATA_DIR, "Results:", RESULTS_DIR)


## Download and checksum verification

Electrical files and the metadata image are retrieved first. Weather is fetched only when INCLUDE_WEATHER is enabled. Downloads are streamed and verified against the pinned Zenodo v2 record.


In [ ]:
names = [
    "dt2_electrical_production_inverter_1_with_faults.csv",
    "dt3_electrical_production_inverter_2.csv",
    "00_metadata_variable.jpg",
]
if INCLUDE_WEATHER:
    names.append("dt1_solar_and_meteorological_measurement.csv")
downloaded = {name: download_file(name, DATA_DIR) for name in names}
display(pd.DataFrame([FILES[name] | {"path": str(path)} for name, path in downloaded.items()]))


## Audit, model, and export

Audits process CSV rows in bounded chunks. Weather is reduced to five-minute bins while streaming. Electrical rows are paired only on exact UTC timestamps; unmatched rows and gaps are never interpolated.

Whole local days define chronological 60/20/20 training, validation, and held-out periods. Healthy validation data select models and freeze the detection floor. Fault labels are only used to exclude experimental observations from fitting and to group documented episodes.


In [ ]:
result = audit_and_model(
    DATA_DIR,
    include_weather=INCLUDE_WEATHER,
    chunksize=CHUNKSIZE,
    produced_at=datetime.now(timezone.utc).replace(microsecond=0).isoformat().replace("+00:00", "Z"),
)
paths = export_results(
    result,
    output_dir=RESULTS_DIR,
    schema_path=ANALYSIS_DIR / "contracts" / "review.schema.v0.1.0.json",
)
print({key: str(path) for key, path in paths.items()})


## Reproduced data-quality audit

In [ ]:
audit_rows = []
for name, audit in result["audits"].items():
    audit_rows.append({
        "file": name, "rows": audit["row_count"],
        "start_utc": audit["timestamp_start_utc"], "end_utc": audit["timestamp_end_utc"],
        "median_interval_s": audit["interval_seconds"]["median"],
        "duplicates": audit["duplicate_timestamp_count"],
        "out_of_order": audit["out_of_order_count"],
        "null_counts": audit["null_counts"], "fault_counts": audit["fault_value_counts"],
    })
display(pd.DataFrame(audit_rows).set_index("file"))
display(pd.Series(result["alignment"], name="Exact inverter timestamp alignment"))
display(pd.Series(result["weather_alignment"], name="Weather overlap"))


## Model comparison and evaluation

In [ ]:
summary = result["summary"]
display(pd.DataFrame(summary["model_selection"]["validation_scores"]).T.rename_axis("candidate"))
display(pd.DataFrame(summary["model_selection"]["heldout_healthy_scores"]).T.rename_axis("candidate"))
print("Progressive validation comparisons:", summary["model_selection"]["progressive_selection_comparisons"])
print("Selected on healthy validation only:", summary["selected_model"]["name"])
print("Chronological periods:", summary["split_metadata"])
print("Held-out healthy calibration:", summary["heldout_healthy"])
print("Post-training shading episodes (validation-period episodes are exploratory):", summary["shortfall"])
print("Final held-out shading only:", summary["heldout_test_shading"])
print("Decision:", summary["decision"])
print("Reference stability:", summary["reference_stability"])
print("Held-out healthy placebo:", summary["placebo"])
print("Missingness sensitivity (synthetic):", summary["missingness_sensitivity"])


## Compact series with source gaps retained

The plot uses nonoverlapping five-minute bins and leaves absent bins blank. It does not forward-fill across gaps. Experimental shading samples appear as a separate mark.


In [ ]:
import matplotlib.pyplot as plt
bins = result["bins"].copy()
model = result["model_info"]["selected_model"]
features = model["features"]
bins["expected_power_w"] = float("nan")
idx = _model_frame(bins, model).index
if len(idx):
    bins.loc[idx, "expected_power_w"] = _predict(model, bins.loc[idx])
bins = bins.set_index("bin_start").sort_index()
full_index = pd.date_range(bins.index.min(), bins.index.max(), freq="5min", tz="UTC")
plot = bins.reindex(full_index)
fig, ax = plt.subplots(figsize=(15, 5))
ax.plot(plot.index, plot["target_power_w"], linewidth=0.7, label="Measured inverter 1")
ax.plot(plot.index, plot["expected_power_w"], linewidth=0.7, label="Healthy reference prediction")
fault = plot["fault_fraction"].fillna(0).gt(0)
ax.scatter(plot.index[fault], plot.loc[fault, "target_power_w"], s=3, color="crimson", label="Documented shading")
ax.set(xlabel="Time (UTC)", ylabel="AC power (W)", title="Five-minute PV series; blank spans are missing source bins")
ax.legend(loc="upper right", ncol=3)
fig.tight_layout()
plt.show()


## Interpretation and reproducibility

Outputs include a computed data-quality report, model comparison JSON, and a review artifact validated against the included provisional v0.1.0 schema when eligible measured episode data are available. Schema mismatches and compatibility mapping are documented in SCHEMA_FEEDBACK.md.

insufficient_evidence can be the correct result when final held-out shading episodes are absent, reference stability fails, or calibration/coverage is inadequate. Validation-period episode estimates remain exploratory. Neither an estimated shortfall nor supported_change identifies a physical cause or verifies commercial energy recovery.
